<a href="https://colab.research.google.com/github/sherjahong1r/Machine-Learning-Lessons/blob/main/08_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Oddiy RNN yaratamiz va shug'ullantiramiz**

## PyTorch bilan RNN yaratish

## salom degan so'z bilan keyingi harfni bashorat qilamiz

## berkitilgan holat (hidden state) oqimini ko'rish













### **vazifalar:**

### kutubxonalarni import qilib olish

### o'yinchoq ma'lumot tayyorlab olish

### model yaratish

### one_hot;chi yaratish

### shug'ullantirish

In [116]:
import torch
import torch.nn as nn
import torch.optim

In [117]:
sequence = "salom"
chars = sorted(list(set(sequence)))
print(chars)

# sequence nomli string o'zgaruvchisini aniqlaydi va unga "salom" qiymatini yuklaydi.

['a', 'l', 'm', 'o', 's']


In [118]:
char2idx = {char: i for i, char in enumerate(chars)}
idx2char = {i: char for i, char in enumerate(chars)}
# char2idx: Har bir noyob harfga uning tartib raqamini (indeksini) belgilaydi.
# Misol uchun, 'a' harfi 0-indeksga, 'b' harfi 1-indeksga va hokazo o'tkaziladi.
# idx2char: Yuqoridagi jarayonning teskarisini qiladi, ya'ni indeksdan unga mos keladigan harfni
# topish imkonini beradi. Misol uchun, 0-indeks 'a' harfiga, 1-indeks 'b' harfiga va hokazo o'tkaziladi.

print(f"Character to index mapping: {char2idx}")
print(f"Index to character mapping: {idx2char}")

Character to index mapping: {'a': 0, 'l': 1, 'm': 2, 'o': 3, 's': 4}
Index to character mapping: {0: 'a', 1: 'l', 2: 'm', 3: 'o', 4: 's'}


In [119]:
x_data = [char2idx[char] for char in sequence[:-1]] # s, a, l, o
y_data = [char2idx[char] for char in sequence[1:]]  # a, l, o, m
# Shu tartibdagi harflarni olib index ga aylantiradi
# Bu modelga keyingi harfni bashorat qilish uchun joriy harflar ketma-ketligini beradi.
print(f"x_data (indexed): {x_data}")
print(f"y_data (indexed): {y_data}")

x_data (indexed): [4, 0, 1, 3]
y_data (indexed): [0, 1, 3, 2]


In [120]:
x = torch.tensor(x_data).unsqueeze(1) #  x_data ni PyTorch tensoriga aylantiradi va model kiritishi uchun unga yangi o'lcham qo'shadi.
y = torch.tensor(y_data) # y_data ni PyTorch tensoriga aylantiradi, bu model bashorat qilishi kerak bo'lgan maqsad qiymatlardir.
x.shape, y.shape #  x va y tensorlarining o'lchamlarini ko'rsatadi.

(torch.Size([4, 1]), torch.Size([4]))

In [121]:
class HarfRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(HarfRNN, self).__init__()
        self.rnn = nn.RNN(vocab_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out.squeeze(1))
        return out, hidden

In [122]:
vocab_size = len(chars)
hidden_size = 8
model = HarfRNN(vocab_size, hidden_size)

In [123]:
def one_hot(index, vocab_size):
  vec = torch.zeros(1, 1, vocab_size)
  vec[0][0][index] = 1
  return vec

In [124]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [125]:
for epoch in range(100):
  loss = 0
  h = torch.zeros(1, 1, hidden_size)

  for i in range(len(x)):
    input_vec = one_hot(x[i].item(), vocab_size)
    output, h = model(input_vec, h.detach())
    loss += criterion(output, y[i].unsqueeze(0))

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoch % 10 ==0:
    pred_seq = ''
    h_test = torch.zeros(1, 1, hidden_size)

    for i in x_data:
      input_vec = one_hot(i, vocab_size)
      output, h_test = model(input_vec, h_test.detach()) # Fixed: added .detach() for h_test
      pred_idx = output.argmax().item()
      pred_char = idx2char[pred_idx]
      pred_seq += pred_char

    print(f"Epoch: {epoch}, Loss: {loss.item()}, Predicted Sequence: {pred_seq}")

Epoch: 0, Loss: 6.464789867401123, Predicted Sequence: aaaa
Epoch: 10, Loss: 4.998533725738525, Predicted Sequence: alom
Epoch: 20, Loss: 3.449925422668457, Predicted Sequence: alom
Epoch: 30, Loss: 1.6960043907165527, Predicted Sequence: alom
Epoch: 40, Loss: 0.6917204260826111, Predicted Sequence: alom
Epoch: 50, Loss: 0.3082956075668335, Predicted Sequence: alom
Epoch: 60, Loss: 0.1741083860397339, Predicted Sequence: alom
Epoch: 70, Loss: 0.11922520399093628, Predicted Sequence: alom
Epoch: 80, Loss: 0.09165523946285248, Predicted Sequence: alom
Epoch: 90, Loss: 0.07513531297445297, Predicted Sequence: alom
